In [1]:
%load_ext autoreload
%autoreload 2
import numpy as np
import gym
from main.alpaca import *
from main.dataset import *
from main.dataViz import *
import yaml
import jax 
import jax.numpy as jnp

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
cfg_filename = 'configs/sinusoid-config.yml'
with open(cfg_filename,'r') as ymlfile:
    config = yaml.load(ymlfile, Loader=yaml.SafeLoader)

In [3]:
noise1 = 0.1
noise2 = 0.3
noise3 = 0.5
dataset1 = SinusoidDataset(config, noise_var=noise1, rng=np.random.RandomState(1234))
dataset2 = SinusoidDataset(config, noise_var=noise2, rng=np.random.RandomState(1234))
dataset3 = SinusoidDataset(config, noise_var=noise3, rng=np.random.RandomState(1234))

## Agents

In [ ]:
steps = 5000
config1 = config.copy()
config1["sigma_eps"] = noise1
agent1 = ALPaCA(config1)
agent1.setup()
state1, train_log1 = train_loop(agent1, dataset1, num_train_updates=steps)
agent1 = TrainedALPaCA(agent1, state1)

2025-08-27 16:34:19.320819: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-27 16:34:19.848835: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-27 16:34:20.226406: W external/xla/xla/service/gpu/autotuning/dot_search_space.cc:200] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs?Working around this by using the full hints set instead.
2025-08-27 16:34:20.573174: W external/xla/xla/service/gpu/au

AttributeError: "ALPaCA" object has no attribute "lr". If "lr" is defined in '.setup()', remember these fields are only accessible from inside 'init' or 'apply'.

In [ ]:
plt.figure(figsize=(6,3))
plt.plot(np.array(train_log1["loss_history"]))
plt.xlabel('Update')
plt.ylabel('Loss')
plt.title('Training loss')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
config2 = config.copy()
config2["sigma_eps"] = noise2
agent2 = ALPaCA(config2)
state2, train_log2 = train_loop(agent2, dataset2, num_train_updates=steps)
agent2 = TrainedALPaCA(agent2, state2)

In [ ]:
config3 = config.copy()
config3["sigma_eps"] = noise3
agent3 = ALPaCA(config3)
state3, train_log3 = train_loop(agent3, dataset3, num_train_updates=steps)
agent3 = TrainedALPaCA(agent3, state3)

# Visualize all agents

In [ ]:
N_test = 50
test_horz = 30
dataset1 = SinusoidDataset(config, noise_var=noise1, rng=np.random.RandomState(4321))
dataset2 = SinusoidDataset(config, noise_var=noise2, rng=np.random.RandomState(4321))
dataset3 = SinusoidDataset(config, noise_var=noise3, rng=np.random.RandomState(4321))
X_test1, Y_test1, freq_list_test, amp_list_test, phase_list_test = dataset1.sample(N_test, test_horz, return_lists=True)
X_test2, Y_test2 = dataset2.sample(N_test, test_horz, return_lists=False)
X_test3, Y_test3 = dataset3.sample(N_test, test_horz, return_lists=False)

In [ ]:
ind = 2
sample_size_list = [0,1,2,3,4,5,6,7,8,9,10]
plt.figure(figsize=(9,len(sample_size_list)*1))
for i,num_pts in enumerate(sample_size_list):
    X_update1 = X_test1[ind:(ind+1),:num_pts,:]
    Y_update1 = Y_test1[ind:(ind+1),:num_pts,:]
    
    X_update2 = X_test2[ind:(ind+1),:num_pts,:]
    Y_update2 = Y_test2[ind:(ind+1),:num_pts,:]
    
    X_update3 = X_test3[ind:(ind+1),:num_pts,:]
    Y_update3 = Y_test3[ind:(ind+1),:num_pts,:]
    
    title=None
    legend=False
    if i == 0:
        legend=True
        title=True
        
    ax1 = plt.subplot(len(sample_size_list),3,3*i+1)
    gen_sin_fig(agent1, X_update1, Y_update1, freq_list_test[ind], phase_list_test[ind], amp_list_test[ind], label=None)
    if i == 0:
        plt.title(r'ALPaCA, ' + r'$\Sigma_\epsilon = 0.1$')
    if i < len(sample_size_list) - 1:
        plt.setp(ax1.get_xticklabels(), visible=False)
    
    ax2 = plt.subplot(len(sample_size_list),3,3*i+2, sharey=ax1)
    gen_sin_fig(agent2, X_update2, Y_update2, freq_list_test[ind], phase_list_test[ind], amp_list_test[ind], label=None)
    plt.setp(ax2.get_yticklabels(), visible=False)
    if i == 0:
        plt.title(r'ALPaCA, ' + r'$\Sigma_\epsilon = 0.3$')
    if i < len(sample_size_list) - 1:
        plt.setp(ax2.get_xticklabels(), visible=False)
    
    
    ax3 = plt.subplot(len(sample_size_list),3,3*i+3, sharey=ax1)
    gen_sin_fig(agent3, X_update3, Y_update3, freq_list_test[ind], phase_list_test[ind], amp_list_test[ind], label=None)
    plt.setp(ax3.get_yticklabels(), visible=False)
    if i == 0:
        plt.title(r'ALPaCA, ' + r'$\Sigma_\epsilon = 0.5$')
    if i < len(sample_size_list) - 1:
        plt.setp(ax3.get_xticklabels(), visible=False)

plt.tight_layout(w_pad=0.0,h_pad=0.2)
plt.savefig('figures/sinusoid_varying_noise.pdf')
plt.show()